# Demo: Analisi emotiva di un singolo commento con ELIta

Questo notebook mostra **passo per passo** come il metodo finale assegna un'emozione a un commento del corpus `r/Italia — notizie`.

**Metodo finale**: lessico ELIta ibrido (α=0.5) + corpus_mean normalisation (Formula 3.5 ItEm)

Per ogni commento selezionato:
1. Si mostrano i token lemmatizzati e quali vengono usati (ADJ/NOUN/VERB trovati in ELIta)
2. Si mostra il contributo emotivo grezzo di ogni parola
3. Si calcola il vettore aggregato e si applica la corpus_mean normalisation
4. Si mostra l'emozione dominante dopo normalizzazione

## Setup e caricamento dati

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from Fase3.support import (
    BASIC_EMOTIONS, EMOTION_COLORS, POS_FILTER,
    load_corpus, load_recalc, compute_mu_e,
    normalizza, emozione_dominante,
    plot_radar_single, plot_raw_vs_norm_bars,
)

CORPUS_CSV   = Path('corpus_Italia_notizie.csv')
TOKENS_CSV   = Path('tokens_Italia_notizie.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')

print('Configurazione caricata.')


Configurazione caricata.


In [2]:
df_corpus, df_tokens = load_corpus(CORPUS_CSV, TOKENS_CSV)
df_elita_final = load_recalc(ALPHA_05_CSV)
mu_e           = compute_mu_e(df_corpus, df_tokens, df_elita_final)

print(f'Corpus: {len(df_corpus)} documenti | Token: {len(df_tokens)}')
print(f'  post: {(df_corpus["type"]=="post").sum()} | commenti: {(df_corpus["type"]=="comment").sum()}')
print()
print('Medie corpus μ_e (ELIta α=0.5):')
for e in BASIC_EMOTIONS:
    print('  {:<15s}: {:.4f}'.format(e, mu_e[e]))

Corpus: 2520 documenti | Token: 84867
  post: 200 | commenti: 2320

Medie corpus μ_e (ELIta α=0.5):
  gioia          : 4.5774
  tristezza      : 3.3630
  rabbia         : 3.3118
  paura          : 3.7476
  disgusto       : 2.4378
  fiducia        : 4.8585
  sorpresa       : 4.5789
  aspettativa    : 5.6053


## Selezione del documento

Modifica `DOC_INDEX` (0–2359) per scegliere un documento diverso, oppure imposta `DOC_ID` con un ID specifico.

Puoi anche filtrare per tipo: `DOC_TYPE = 'comment'` mostra solo commenti, `DOC_TYPE = 'post'` solo post, `None` per tutti.

In [3]:
DOC_INDEX = 16       # <--- cambia qui (0-2359)
DOC_ID    = None     # oppure specifica un ID, es. 'kfr4gvl' (commento) o '1idmjsb' (post)
DOC_TYPE  = 'comment'  # 'comment', 'post', oppure None per tutti

# Filtra per tipo se richiesto
df_sel = df_corpus[df_corpus['type'] == DOC_TYPE] if DOC_TYPE else df_corpus
df_sel = df_sel.reset_index(drop=True)

if DOC_ID:
    row = df_sel[df_sel['doc_id'] == DOC_ID].iloc[0]
else:
    row = df_sel.iloc[DOC_INDEX]

CID  = row['doc_id']
TEXT = row['text']

print(f'ID          : {CID}')
print(f'Tipo        : {row["type"]}')
print(f'Autore      : {row["author"]}')
print(f'Score Reddit: {row["score"]}')
print()
print('Testo:')
print('-' * 70)
print(TEXT)
print('-' * 70)

ID          : muot2my
Tipo        : comment
Autore      : No_Pen_469
Score Reddit: 1

Testo:
----------------------------------------------------------------------
E poi magari trovi il modo di leggerli per vie "totalmente legali" ed è un articolo inutilmente lungo di cose quasi a caso senza la vera informazione che stava scritta sul titolo. Questo pure prima di chatgpt.
----------------------------------------------------------------------


## Token e lemmi del commento

In [4]:
df_tok = df_tokens[df_tokens['doc_id'] == CID].copy()

elita_idx_all = set(df_elita_final.index)

df_tok['in_ELIta'] = df_tok['lemma'].isin(elita_idx_all)
df_tok['pos_ok']   = df_tok['pos'].isin(POS_FILTER)
df_tok['usato']    = df_tok['in_ELIta'] & df_tok['pos_ok']

print(f'Token totali nel documento     : {len(df_tok)}')
print(f'Con POS valida (ADJ/NOUN/VERB) : {df_tok["pos_ok"].sum()}')
print(f'Trovati in ELIta               : {df_tok["in_ELIta"].sum()}')
print(f'Token usati per l\'analisi      : {df_tok["usato"].sum()}')
print()

display(df_tok[['token','lemma','pos','in_ELIta','usato']].reset_index(drop=True))

Token totali nel documento     : 37
Con POS valida (ADJ/NOUN/VERB) : 15
Trovati in ELIta               : 15
Token usati per l'analisi      : 12



,token,lemma,pos,in_ELIta,usato
0,E,e,CCONJ,False,False
1,poi,poi,ADV,False,False
2,magari,magari,ADV,True,False
3,trovi,trovare,VERB,True,True
4,il,il,DET,False,False
5,modo,modo,NOUN,True,True
6,di,di,ADP,False,False
7,leggerli,leggere li,VERB,False,False
8,per,per,ADP,False,False
9,vie,via,NOUN,True,True


## Contributo emotivo per parola

Per ogni lemma usato nell'analisi, mostriamo il vettore emotivo da ELIta α=0.5 (score grezzi).

In [5]:
lemmi_usati = df_tok[df_tok['usato']]['lemma'].tolist()

if not lemmi_usati:
    print('Nessun lemma utile trovato in questo commento.')
else:
    contrib_rows = []
    for lemma in lemmi_usati:
        scores = df_elita_final.loc[lemma, BASIC_EMOTIONS].to_dict()
        dom    = max(scores, key=scores.get)
        scores['lemma']   = lemma
        scores['dom_emo'] = dom
        contrib_rows.append(scores)

    df_contrib = pd.DataFrame(contrib_rows)
    cols_show  = ['lemma'] + BASIC_EMOTIONS + ['dom_emo']

    totals  = df_contrib[BASIC_EMOTIONS].sum()
    dom_raw = totals.idxmax()

    print('Contributi emotivi per lemma (ELIta α=0.5 — score grezzi):')
    display(
        df_contrib[cols_show]
        .style
        .background_gradient(subset=BASIC_EMOTIONS, cmap='YlOrRd', axis=None)
        .format({e: '{:.2f}' for e in BASIC_EMOTIONS})
    )
    print()
    print('Score aggregato grezzo (S_e):')
    print(totals.round(3).to_string())
    print(f'\n=> Emozione dominante (raw): {dom_raw.upper()}')

Contributi emotivi per lemma (ELIta α=0.5 — score grezzi):


,lemma,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa,dom_emo
0,trovare,0.78,0.21,0.26,0.40,0.17,0.73,0.92,0.87,sorpresa
1,modo,0.15,0.08,0.09,0.12,0.06,0.21,0.35,0.48,aspettativa
2,via,0.59,0.69,0.56,0.56,0.37,0.26,0.32,0.41,tristezza
3,legale,0.24,0.22,0.31,0.43,0.19,0.60,0.44,0.47,fiducia
4,articolo,0.41,0.08,0.08,0.12,0.06,0.55,0.42,0.65,aspettativa
5,cosa,0.36,0.55,0.38,0.37,0.51,0.33,0.38,0.32,tristezza
6,caso,0.45,0.42,0.35,0.44,0.27,0.43,0.82,0.46,sorpresa
7,vero,0.77,0.32,0.32,0.22,0.17,0.82,0.57,0.67,fiducia
8,informazione,0.67,0.51,0.39,0.44,0.30,0.46,0.64,0.75,aspettativa
9,stare,0.60,0.36,0.21,0.25,0.19,0.57,0.49,0.71,aspettativa



Score aggregato grezzo (S_e):
gioia          5.747
tristezza      3.812
rabbia         3.402
paura          3.929
disgusto       2.690
fiducia        5.806
sorpresa       6.467
aspettativa    7.129

=> Emozione dominante (raw): ASPETTATIVA


## Corpus_mean normalisation (Formula 3.5 ItEm)

Lo score grezzo viene diviso per la media di corpus μ_e:

```
S_e_norm(d) = S_e(d) / μ_e
```

Le emozioni con valore medio alto nel corpus (come aspettativa) vengono penalizzate proporzionalmente.

In [6]:
if not lemmi_usati:
    print('Nessun lemma utile trovato.')
else:
    sc_norm  = normalizza(totals.to_dict(), mu_e)
    dom_norm = emozione_dominante(sc_norm)

    print('Corpus_mean normalisation:')
    print('{:<15s} {:>10s} {:>10s} {:>10s}'.format('Emozione', 'S_e (raw)', 'μ_e', 'S_e_norm'))
    print('-' * 50)
    for e in BASIC_EMOTIONS:
        print('{:<15s} {:>10.3f} {:>10.3f} {:>10.3f}'.format(
            e, totals[e], mu_e[e], sc_norm[e]))
    print(f'\n=> Emozione dominante (corpus_mean): {dom_norm.upper()}')

    plot_radar_single(
        sc_norm, dom_norm,
        title=f'Profilo emotivo normalizzato — commento {CID} (metodo finale)',
        height=480,
    ).show()


Corpus_mean normalisation:
Emozione         S_e (raw)        μ_e   S_e_norm
--------------------------------------------------
gioia                5.747      4.577      1.255
tristezza            3.812      3.363      1.133
rabbia               3.402      3.312      1.027
paura                3.929      3.748      1.048
disgusto             2.690      2.438      1.103
fiducia              5.806      4.859      1.195
sorpresa             6.467      4.579      1.412
aspettativa          7.129      5.605      1.272

=> Emozione dominante (corpus_mean): SORPRESA


## Confronto: score grezzo vs corpus_mean normalizzato

In [7]:
if not lemmi_usati:
    print('Nessun lemma utile trovato.')
else:
    plot_raw_vs_norm_bars(totals.to_dict(), sc_norm, CID).show()

    print(f'Lemmi usati ({len(lemmi_usati)}): {lemmi_usati}')
    print(f'Emozione dominante raw          : {dom_raw.upper()}  (score: {totals[dom_raw]:.3f})')
    print(f'Emozione dominante corpus_mean  : {dom_norm.upper()}  (score: {sc_norm[dom_norm]:.3f})')


Lemmi usati (12): ['trovare', 'modo', 'via', 'legale', 'articolo', 'cosa', 'caso', 'vero', 'informazione', 'stare', 'scritta', 'titolo']
Emozione dominante raw          : ASPETTATIVA  (score: 7.129)
Emozione dominante corpus_mean  : SORPRESA  (score: 1.412)
